Training On GPU


In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim

In [22]:
torch.manual_seed(42)

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Unsig Device: {device}")

Unsig Device: cuda


In [24]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
df = pd.read_csv("/content/drive/MyDrive/DataSets/fashion-mnist_train.csv")

In [26]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [37]:
df.shape

(60000, 785)

In [28]:
X = df.drop(columns="label")
y = df["label"]

In [29]:
X_train, X_test, y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [30]:
X_train = X_train/255.0
X_test = X_test/255.0

In [31]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features.to_numpy(),dtype=torch.float32)
        self.labels = torch.tensor(labels.to_numpy(),dtype=torch.long)

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self,idx):
        return self.features[idx], self.labels[idx]

In [32]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)

In [33]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [34]:
class MyANN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )

    def forward(self,features):
        return self.model(features)

In [35]:
learning_rate = 0.1
epochs = 100

In [38]:
# instantiate model
model = MyANN(X_train.shape[1])
model = model.to(device)

# loss function
criterion = nn.CrossEntropyLoss()

# optimizer 
optimizer = optim.SGD(model.parameters(),lr=learning_rate)

In [39]:
# training loop
for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features, batch_labels in train_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # forward pass
        outputs = model(batch_features)

        # calculate loss
        loss = criterion(outputs,batch_labels)

        # back pass
        optimizer.zero_grad()
        loss.backward()

        # gradients update
        optimizer.step()

        # total epoch loss
        total_epoch_loss = total_epoch_loss + loss.item()

    # avg loss
    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch: {epoch+1}, Loss: {avg_loss}")

Epoch: 1, Loss: 0.6352872474888961
Epoch: 2, Loss: 0.4304986953884363
Epoch: 3, Loss: 0.3861262078657746
Epoch: 4, Loss: 0.3584607255011797
Epoch: 5, Loss: 0.3376494748592377
Epoch: 6, Loss: 0.32276468626906474
Epoch: 7, Loss: 0.3078539018382629
Epoch: 8, Loss: 0.2949818898836772
Epoch: 9, Loss: 0.2854692505300045
Epoch: 10, Loss: 0.27467058210571604
Epoch: 11, Loss: 0.26830569267148774
Epoch: 12, Loss: 0.2581421597401301
Epoch: 13, Loss: 0.24940819991752505
Epoch: 14, Loss: 0.24444738873218497
Epoch: 15, Loss: 0.2385919222868979
Epoch: 16, Loss: 0.23155899402375021
Epoch: 17, Loss: 0.22562562982489665
Epoch: 18, Loss: 0.2202964697740972
Epoch: 19, Loss: 0.21206334652379155
Epoch: 20, Loss: 0.20960091769819458
Epoch: 21, Loss: 0.20624992956593632
Epoch: 22, Loss: 0.19986103242821993
Epoch: 23, Loss: 0.1953041393881043
Epoch: 24, Loss: 0.19312163671137145
Epoch: 25, Loss: 0.18764107141271233
Epoch: 26, Loss: 0.18366442496639987
Epoch: 27, Loss: 0.18018299543857574
Epoch: 28, Loss: 0.173

In [40]:
# set model to eval mode
model.eval()

MyANN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [43]:
# evaluation code
total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:

        # move data to gpu
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        outputs = model(batch_features)
        _, predicted = torch.max(outputs,1)

        total = total + batch_labels.shape[0]
        correct = correct + (predicted == batch_labels).sum().item()

    print(correct/total)
    


0.8898333333333334
